In [ ]:
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import numpy as np


df = pd.read_csv('../data/raw/santander-customer-transaction-prediction/train.csv')
X = df.drop(["target", "ID_code"], axis=1)
y = df["target"]

In [2]:
from sklearn.model_selection import StratifiedKFold, train_test_split

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [ ]:
test_df = pd.read_csv('../data/raw/santander-customer-transaction-prediction/test.csv')
X_test = test_df.drop("ID_code", axis=1)


def run_cv(min_child_samples):

    test_preds = np.zeros(len(X_test))
    oof_preds = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_auc_scores = []
    best_iterations = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
            n_estimators=1000,
            learning_rate=0.05,
            random_state=42,
            num_leaves=16,
            min_child_samples=min_child_samples
        )
        
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=100)
            ]
        )
        
        fold_preds = model.predict_proba(X_val_fold)[:, 1]
        fold_auc = roc_auc_score(y_val_fold, fold_preds)
        fold_auc_scores.append(fold_auc)
        best_iterations.append(model.best_iteration_)

        oof_preds[val_idx] = fold_preds
        test_preds += (
            model.predict_proba(
                X_test,
                num_iteration=model.best_iteration_
            )[:, 1]
            / skf.n_splits
        )
        
        print(f"Fold {fold + 1} AUC: {fold_auc:.6f}")
        oof_auc = roc_auc_score(y, oof_preds)

    return {
        "min_child_samples": min_child_samples,
        "mean_auc": np.mean(fold_auc_scores),
        "oof_auc": oof_auc,
        "std_auc": np.std(fold_auc_scores),
        "mean_best_iteration": np.mean(best_iterations),
        "best_iterations": best_iterations
    }

In [4]:
results = []

for min_child_samples in [20, 50, 100, 250, 500]:
    results.append(run_cv(min_child_samples))

[LightGBM] [Info] Number of positive: 16079, number of negative: 143921
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.062726 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 200
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100494 -> initscore=-2.191750
[LightGBM] [Info] Start training from score -2.191750
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.838228	valid_0's binary_logloss: 0.268478
[200]	valid_0's auc: 0.863976	valid_0's binary_logloss: 0.246815
[300]	valid_0's auc: 0.876273	valid_0's binary_logloss: 0.234152
[400]	valid_0's auc: 0.883314	valid_0's binary_logloss: 0.22596
[500]	valid_0's auc: 0.887499	valid_0's binary_logloss: 0.220263
[600]	valid_0's auc: 0.89046	valid_0's binary_logloss: 0.216071
[700]	valid_0's auc: 0.892522	valid_0's binary_loglo

In [5]:

for res in results:
    print(
        f"min_child_samples={res['min_child_samples']}, "
        f"mean_auc={res['mean_auc']:.6f}, "
        f"std_auc={res['std_auc']:.6f}, "
        f"oof_auc={res['oof_auc']:.6f}, "
        f"mean_best_iteration={res['mean_best_iteration']:.6f}, "
        f"best_iterations={res['best_iterations']}"
    )

min_child_samples=20, mean_auc=0.892819, std_auc=0.002968, oof_auc=0.892826, mean_best_iteration=986.400000, best_iterations=[999, 994, 960, 979, 1000]
min_child_samples=50, mean_auc=0.892920, std_auc=0.003178, oof_auc=0.892918, mean_best_iteration=979.000000, best_iterations=[969, 995, 981, 968, 982]
min_child_samples=100, mean_auc=0.892963, std_auc=0.002935, oof_auc=0.892960, mean_best_iteration=991.800000, best_iterations=[993, 997, 970, 999, 1000]
min_child_samples=250, mean_auc=0.893294, std_auc=0.002906, oof_auc=0.893283, mean_best_iteration=997.800000, best_iterations=[997, 999, 996, 998, 999]
min_child_samples=500, mean_auc=0.893896, std_auc=0.002652, oof_auc=0.893895, mean_best_iteration=994.400000, best_iterations=[1000, 991, 1000, 990, 991]


In [6]:
results1 = []

for min_child_samples in [500, 1000, 2000, 5000]:
    results1.append(run_cv(min_child_samples))


for res in results1:
    print(
        f"min_child_samples={res['min_child_samples']}, "
        f"mean_auc={res['mean_auc']:.6f}, "
        f"std_auc={res['std_auc']:.6f}, "
        f"oof_auc={res['oof_auc']:.6f}, "
        f"mean_best_iteration={res['mean_best_iteration']:.6f}, "
        f"best_iterations={res['best_iterations']}"
    )

[LightGBM] [Info] Number of positive: 16079, number of negative: 143921
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.072592 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 200
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100494 -> initscore=-2.191750
[LightGBM] [Info] Start training from score -2.191750
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.83753	valid_0's binary_logloss: 0.268625
[200]	valid_0's auc: 0.863505	valid_0's binary_logloss: 0.247341
[300]	valid_0's auc: 0.876089	valid_0's binary_logloss: 0.234534
[400]	valid_0's auc: 0.883265	valid_0's binary_logloss: 0.226103
[500]	valid_0's auc: 0.887638	valid_0's binary_logloss: 0.220143
[600]	valid_0's auc: 0.890682	valid_0's binary_logloss: 0.215785
[700]	valid_0's auc: 0.892631	valid_0's binary_logl

In [3]:
def run_cv(test_preds, min_child_samples, n_estimators):
    oof_preds = np.zeros(len(X))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    fold_auc_scores = []
    best_iterations = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y.iloc[train_idx], y.iloc[val_idx]
        
        model = lgb.LGBMClassifier(
            n_estimators=n_estimators,
            learning_rate=0.05,
            random_state=42,
            num_leaves=16,
            min_child_samples=min_child_samples
        )
        
        model.fit(
            X_train_fold, y_train_fold,
            eval_set=[(X_val_fold, y_val_fold)],
            eval_metric="auc",
            callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=100)
            ]
        )
        
        fold_preds = model.predict_proba(X_val_fold)[:, 1]
        fold_auc = roc_auc_score(y_val_fold, fold_preds)
        fold_auc_scores.append(fold_auc)
        best_iterations.append(model.best_iteration_)

        oof_preds[val_idx] = fold_preds
        test_preds += (
            model.predict_proba(
                X_test,
                num_iteration=model.best_iteration_
            )[:, 1]
            / skf.n_splits
        )
        
        print(f"Fold {fold + 1} AUC: {fold_auc:.6f}")
        oof_auc = roc_auc_score(y, oof_preds)

    return {
        "test_preds": test_preds,
        "min_child_samples": min_child_samples,
        "n_estimators": n_estimators,
        "mean_auc": np.mean(fold_auc_scores),
        "oof_auc": oof_auc,
        "std_auc": np.std(fold_auc_scores),
        "mean_best_iteration": np.mean(best_iterations),
        "best_iterations": best_iterations
    }

In [6]:
test_preds = np.zeros(len(X_test))
res = run_cv(test_preds, min_child_samples=5000, n_estimators=5000)

print(
    f"min_child_samples={res['min_child_samples']}, "
    f"mean_auc={res['mean_auc']:.6f}, "
    f"std_auc={res['std_auc']:.6f}, "
    f"oof_auc={res['oof_auc']:.6f}, "
    f"mean_best_iteration={res['mean_best_iteration']:.6f}, "
    f"best_iterations={res['best_iterations']}"
)

[LightGBM] [Info] Number of positive: 16079, number of negative: 143921
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.062148 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 51000
[LightGBM] [Info] Number of data points in the train set: 160000, number of used features: 200
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.100494 -> initscore=-2.191750
[LightGBM] [Info] Start training from score -2.191750
Training until validation scores don't improve for 100 rounds
[100]	valid_0's auc: 0.841169	valid_0's binary_logloss: 0.267817
[200]	valid_0's auc: 0.866253	valid_0's binary_logloss: 0.245942
[300]	valid_0's auc: 0.878123	valid_0's binary_logloss: 0.232939
[400]	valid_0's auc: 0.884514	valid_0's binary_logloss: 0.224594
[500]	valid_0's auc: 0.888611	valid_0's binary_logloss: 0.21889
[600]	valid_0's auc: 0.891415	valid_0's binary_logloss: 0.214513
[700]	valid_0's auc: 0.893498	valid_0's binary_logl

In [7]:
submission = pd.read_csv("../data/raw/santander-customer-transaction-prediction/sample_submission.csv")
submission["target"] = res["test_preds"]
submission.to_csv("../submissions/exp005.csv", index=False)